In [1]:
%load_ext autoreload
%autoreload 2

# Labelled examples

This exploratory notebook serves for the creation of labelled examples

In [2]:
import pandas as pd
import json
from collections import Counter
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import re
import pycountry
from text_processing_functions import *
from LLM_functions import *
from plot_functions import *
import copy as cp

ERROR 1: PROJ: proj_create_from_database: Open of /home/lhasbini/.conda/envs/dev/share/proj failed


In [3]:
###### FILE PATHS 
fig_path = '/home/lhasbini/como_school/figure/'
file_path_save = '/scratchx/lhasbini/como_school/'
json_file = '/scratchx/lhasbini/como_school/filtered_report_types_nat_hazards_summary-header.json'
all_json = '/scratchx/lhasbini/como_school/all535_predict_llmgpt-4o-mini.json'

##### Open and read the JSON file
with open(json_file, 'r') as json_file:
    filtered_reports = json.load(json_file)

#### EXAMPLE OF LABELLED REPORTS
reports_labelled = pd.read_csv(file_path_save+"labelled_example.csv", encoding='utf-8')

In [4]:
reports_labelled

,Hazard,Country,Locations,Start_Date,End_Date,Name,appealCode
0,Flood,DZA,"['southern and western Algeria', 'Bchar, Elbay...","September 5, 2024","September 8, 2024",NaN,MDRDZ011
1,Flood,PAK,"['Balochistan ', 'Sindh ', 'Punjab', 'Khyber P...",July 2024,NaN,NaN,MDRPK026
2,Mass movement,PAK,"['KP, Azad Jammu and Kashmir AJK, and GilgitBa...",NaN,NaN,NaN,MDRPK026
3,Heat Wave,PAK,NaN,26 August 2024,1 September 2024,NaN,MDRPK026
4,Flood,CMR,"['Cameroons Far North region', 'Logone et Char...",second half of July 2024,"August 28, 2024",NaN,MDRCM039
5,Drought,CMR,NaN,2024,NaN,NaN,MDRCM039
6,Flood,BEN,"['Mono, Couffo, Zou and Oum in the South of Be...",26 June 2024,NaN,NaN,MDRBJ019
7,Flood,SDN,"['Red Sea, River Nile, and Northern State']",1 June 2024,12 August 2024,NaN,MDRSD034
8,Flood,NGA,"['Kano', 'Maiduguri', 'Bauchi state', 'Bauchi,...",8 August 2024,13 August 2024,NaN,MDRNG041
9,Flood,NGA,"['Sokoto State', 'Dantudu, Balakozo, Gidan Tud...",17 July 2024,NaN,NaN,MDRNG041


## Report labelling

Choose a hazard directory. In the case below we use : \
hazard_all_subtype_emdat = {
“drought”, 
“forest fire”, “land fire”, 
“ground movement”, “tsunami”, 
“avalanche”, “landslide”, “rockfall”, “sudden subsidence”, “mudslide", 
“ash fall”, “lava flow”, “pyroclastic flow”, “lahar”, 
“coastal flood”, “flash flood”, “riverine flood”, “ice jam flood”,
“rogue wave”, “seiche”, 
”coldwave”, “heatwave”, “severe winter conditions”, 
“derecho”, “hail”, “lightning/thunderstorm”, “sand/dust storm”,  “winter storm/blizzard”, “storm surge”, “tornado”, “extra-tropical storm”, “tropical cyclone”
}

The flood subtype being hard to differentiate, we will assign hazard to "flash flood" when the text mention heavy rain, "riverine flood" if nothing specific is mentioned. 


Choose a hazard dict : 
hazard_subtype_emdat = {
'Drought': r"drought.",
'Wildfire': r"wildfire.|forest fire.|land fire." , 
‘Earthquake’ : r”ground movement.|tsunami.”, 
‘Mass movement’: r"avalanche.|landslide.|rockfall.|sudden subsidence.|mudslide.",
‘Volcanic activity’ : r”“ash fall.|lava flow.|pyroclastic flow.|lahar”
'Flood': r"\b(coastal flood.|flash flood.|riverine flood.|ice jam flood.)\b",
‘Wave action’ : r“rogue wave.|seiche”,
‘Extreme temperature’ : r”coldwave.|heatwave.|severe winter conditions.”, 
‘Storm’ : r”derecho.|hail.|lightning.|winterstorm.|storm surge.|tornado.|winter storm.|extra-tropical storm.|tropical storm.”
}


In [6]:
filtered_reports[0]['header']

['DREF Operation Algeria Flood 2024 Bechar Appeal MDRDZ011 Country Algeria Hazard Flood Type of DREF Response Crisis Category Yellow Event Onset Sudden DREF Allocation CHF 499,186 Glide Number FL2024000168DZA People Affected 11,100 people People Targeted 6,000 people Operation Start Date 19092024 Operation Timeframe 6 months Operation End Date 31032025 DREF Published 22092024 Targeted Areas Bchar, Tamanrasset, El Bayadh Page 1 13 Description of the Event Date of event 08092024 What happened, where and when?',
 'On September 8, 2024, a severe tropical disturbance triggered widespread flooding across several provinces in southern and western Algeria.',
 'The most affected areas include Bchar, Elbayadh, Beni Abbes, Tamanrasset, Tiaret, Tindouf, and Naama.',
 'The flooding, which began around September 5th, intensified by September 8th, displacing approximately 2,220 families, some of them from nomadic communities.',
 'In Bchar, the number of displaced families surged to 2,060, which were 

In [7]:
dict_0=[
    {
      "Hazard": "Flood",
      "Country" : "Algeria", 
      "Locations": ["southern and western Algeria", "Bchar, Elbayadh, Beni Abbes, Tamanrasset, Tiaret, Tindouf, and Naama"],
      "Start_Date": "September 5, 2024",
      "End_Date": "September 8, 2024"
    }
]
df_0 = pd.DataFrame(dict_0)
df_0['appealCode'] = filtered_reports[0]['appealCode']
df_0['Country'] = [country_name_to_iso3(cntr) for cntr in df_0['Country']]

In [8]:
filtered_reports[1]['header']

['Map IFRC, IM What happened, where and when?',
 'Pakistan has experienced an unusually intense and prolonged monsoon season, resulting in widespread infrastructure damage, numerous casualties, and significant injuries.',
 'The season, which began in July 2024, continued through August, with particularly heavy rainfall recorded throughout the month.',
 'The latest significant weather spell, from 26 August to 1 September 2024, exacerbated the situation.',
 'During this period, the Pakistan Meteorological Department PMD issued forecasts of additional heavy rainfall, worsening the already critical conditions.',
 'The monsoon rains have been exceptionally severe, with rainfall levels reaching up to 318 per cent above normal in some areas.',
 'Regionally, Balochistan received 239 per cent more rainfall than usual, Sindh 318 per cent, Punjab 111 per cent, and Khyber Pakhtunkhwa KP 25 per cent.',
 'This unprecedented volume of rainfall, coupled with unusually high temperatures, accelerated sn

In [9]:
dict_1 = [
    {
        "Hazard": "Flood",
        "Country": "Pakistan",
        "Locations": ["Balochistan ", "Sindh ", "Punjab", "Khyber Pakhtunkhwa KP", "Azad Jammu", "Khyber Pakhtunkhwa KP", 
                     "Jacobabad, Naushahro Feroz, Ghotki, Sukkur, Sanghar, Dadu, Shaheed Benazirabad, and Kashmor", 
                     "Taluka Tando Adam"],
        "Start_Date": "July 2024",
        "End_Date": "NULL",
    },
    {
        "Hazard": "Mass movement",
        "Country": "Pakistan",
        "Locations": ["KP, Azad Jammu and Kashmir AJK, and GilgitBaltistan GB"],
        "Start_Date": "NULL",
        "End_Date": "NULL",
    },
    {
        "Hazard": "Extreme temperature",
        "Country": "Pakistan",
        "Locations": "NULL",
        "Start_Date": "26 August 2024",
        "End_Date": "1 September 2024",
    }
]
df_1 = pd.DataFrame(dict_1)
df_1['appealCode'] = filtered_reports[1]['appealCode']
df_1['Country'] = [country_name_to_iso3(cntr) for cntr in df_1['Country']]

In [10]:
#filtered_reports[2]['appealCode']
filtered_reports[2]['header']

['Appeal MDRCM039 Country Cameroon Hazard Flood Type of DREF Response Crisis Category Yellow Event Onset Slow DREF Allocation CHF 421,471 Glide Number People Affected 158,620 people People Targeted 4,800 people Operation Start Date 04092024 Operation Timeframe 5 months Operation End Date 28022025 DREF Published 13092024 Targeted Areas ExtrmeNord Page 1 19 Description of the Event Date when the trigger was met 28082024 Carte de lExtrme Nord What happened, where and when?',
 'Cameroons Far North region has been experiencing flooding since the start of the rainy season, which began in the second half of July with an average rainfall frequency of one day out of four.',
 'The intensification and recurrence of rains starting from August 10, 2024, has led to a progressive increase in rainfall levels between August 10 and August 19, 2024.',
 'Series of floods have been recorded since August 19, reaching critical levels in the Logone et Chari and Mayo Danay divisions between August 11 and 21, 2

In [11]:
dict_2 = [
    {
        "Hazard": "Flood",
        "Country": "Cameroon",
        "Locations": ["Cameroons Far North region", "Logone et Chari and Mayo Danay", "Yagoua", "Blangoua, Mackary, and Zina", 
                      "Chari division", "Maga, Yagoua", "Logone division", "Ndoukoula district"],
        "Start_Date": "second half of July 2024",
        "End_Date": "August 28, 2024",
    },
    {
        "Hazard": "Drought",
        "Country": "Cameroon",
        "Locations": "NULL",
        "Start_Date": "2024",
        "End_Date": "NULL",
    }
]

df_2 = pd.DataFrame(dict_2)
df_2['appealCode'] = filtered_reports[2]['appealCode']
df_2['Country'] = [country_name_to_iso3(cntr) for cntr in df_2['Country']]

In [12]:
#filtered_reports[3]['appealCode']
filtered_reports[3]['header']

['DREF Operation BeninFlood in Lalo Field visits to displaced communities hosted in a school in Couffo RCB Appeal MDRBJ019 Country Benin Hazard Flood Type of DREF Response Crisis Category Yellow Event Onset Sudden DREF Allocation CHF 254,682 Glide Number People Affected 34,052 people People Targeted 10,215 people Operation Start Date 12072024 Operation Timeframe 4 months Operation End Date 30112024 DREF Published 23072024 Targeted Areas Couffo Page 1 17 Description of the Event Date of event 26062024 MAP most affected district by Red Cross of Benin What happened, where and when?',
 'Intense rainfall observed in the departments of Mono, Couffo, Zou and Oum in the South of Benin caused the overflow of the river Couffo on 26 June 2024 in 6 of the 11 districts of the commune it crosses in Couffo department, Adoukandji, Ahomadegbe, Gnizounme, Tchito, Tohou and Zalli.',
 'A rapid assessment conducted during the following days by Benin Red Cross and the Lalo council on July 1, 2024 indicates 

In [13]:
dict_3 = [
    {
        "Hazard": "Flood",
        "Country": "Benin",
        "Locations": ["Mono, Couffo, Zou and Oum in the South of Benin", "Couffo department, Adoukandji, Ahomadegbe, Gnizounme, Tchito, Tohou and Zalli", 
                      "Ahouada, Hazin, Yamontou, Ahomadegbe, Gnizounme, Hangbannou, Tandji, Aboti, Zounhome, Hehokpa, Sawanou, Tohou Centre and Adjassagon"
                     ],
        "Start_Date": "26 June 2024",
        "End_Date": "NULL",
    }
]

df_3 = pd.DataFrame(dict_3)
df_3['appealCode'] = filtered_reports[3]['appealCode']
df_3['Country'] = [country_name_to_iso3(cntr) for cntr in df_3['Country']]

In [14]:
#print(filtered_reports[4]['appealCode'])
print(filtered_reports[4]['header'])

['IFRC, IM What happened, where and when?', 'Sudan has been grappling with heavy rains that have led to widespread flooding across many regions, worsening the already dire situation caused by the conflict that began 16 months ago.', 'Between June 1 and August 12, 2024, DTM Sudan reported 60 incidents of heavy rains and floods, resulting in sudden displacement IOM.', 'The rainy season is expected to continue until October 2024, with forecasts predicting aboveaverage rainfall.', 'The likelihood of additional flash floods and river flooding remains high.', 'So far, Red Sea, River Nile, and Northern State have been the most severely affected.', 'Sudan has faced flooding in recent years, but the current humanitarian crisis, combined with heavy rains and deadly floods, has had a devastating impact on communities.', 'Both those displaced by conflict and the host communities supporting them under already challenging conditions are suffering.', 'The floods have rendered roads impassable, furthe

In [15]:
dict_4 = [
    {
        "Hazard": "Flood",
        "Country": "Sudan",
        "Locations": ["Red Sea, River Nile, and Northern State" 
                     ],
        "Start_Date": "1 June 2024",
        "End_Date": "12 August 2024",
    }
]
df_4 = pd.DataFrame(dict_4)
df_4['appealCode'] = filtered_reports[4]['appealCode']
df_4['Country'] = [country_name_to_iso3(cntr) for cntr in df_4['Country']]

In [16]:
#print(filtered_reports[5]['appealCode'])
print(filtered_reports[5]['header'])

['DREF Operation Nigeria Floods DREF 2024 Flood cuts off major access road linking Kano to Maiduguri in Katagum community, Bauchi state Appeal MDRNG041 Country Nigeria Hazard Flood Type of DREF Response Crisis Category Yellow Event Onset Sudden DREF Allocation CHF 231,293 Glide Number People Affected 50,000 people People Targeted 9,000 people Operation Start Date 03092024 Operation Timeframe 4 months Operation End Date 31012025 DREF Published 06092024 Targeted Areas Bauchi, Kebbi, Sokoto, Zamfara Page 1 17 Description of the Event Date of event 13082024 Nigeria Flood Forecast 2024 What happened, where and when?', 'From August 8 to August 13, 2024, continuous heavy rainfall triggered severe flooding across Nigeria, leading to widespread devastation and displacement in states such as Bauchi, Sokoto, and Zamfara.', 'In Bauchi State, over 1,000 homes were destroyed, particularly impacting the Giade, Shira, and Katagum local government areas.', 'Earlier, on July 17, 2024, flooding in Sokoto

In [17]:
dict_5 = [
    {
        "Hazard": "Flood",
        "Country": "Nigeria",
        "Locations": ["Kano", "Maiduguri", "Bauchi state", "Bauchi, Kebbi, Sokoto, Zamfara"
                     ],
        "Start_Date": "8 August 2024",
        "End_Date": "13 August 2024",
    }, 
    {
        "Hazard": "Flood",
        "Country": "Nigeria",
        "Locations": ["Sokoto State", "Dantudu, Balakozo, Gidan Tudu, and Tsitse", "Zamfara State", "Ruwan Gora, Morai, Makera, and Talata Mafara town"],
        "Start_Date": "17 July 2024",
        "End_Date": "NULL",
    }
]
df_5 = pd.DataFrame(dict_5)
df_5['appealCode'] = filtered_reports[5]['appealCode']
df_5['Country'] = [country_name_to_iso3(cntr) for cntr in df_5['Country']]

In [18]:
#print(filtered_reports[6]['appealCode'])
print(filtered_reports[6]['header'])

['Page 1 23 Description of the Event Map of the areas most affected by the disaster Date of event 06052023 What happened, where and when?', 'From 1 to 6 May, Rwanda experienced continuous torrential rains, which caused major damage in several Districts of the country.', 'According to assessments carried out by the Rwanda Red Cross and other stake holders and MINEMA led, the western, northern and southern provinces of Rwanda were the areas hardest hit by the flooding.', 'Overall, 14 districts experienced flooding and landslides affecting around 51,905 people in 10,381 households.', 'A total of 137 people died, and 5,472 houses were destroyed.', 'Damage reported includes major losses of houses, basic household items, unusable water sources, latrines and roads.', 'The destruction of thousands of hectares of crops and livestock was immense.', 'Those affected were gathered together in IDP sites.', 'The needs were enormous and the vulnerabilities high.', 'The rains continued until June 2023.

In [19]:
dict_6 = [
    {
        "Hazard": "Flood",
        "Country": "Rwanda",
        "Locations": ["western, northern and southern provinces", "14 districts"
                     ],
        "Start_Date": "1 May 2023",
        "End_Date": "June 2023",
    }, 
    {
        "Hazard": "Mass movement",
        "Country": "Rwanda",
        "Locations": ["14 districts"
                     ],
        "Start_Date": "1 May 2023",
        "End_Date": "NULL",
    }, 
]
df_6 = pd.DataFrame(dict_6)
df_6['appealCode'] = filtered_reports[6]['appealCode']
df_6['Country'] = [country_name_to_iso3(cntr) for cntr in df_6['Country']]

In [20]:
#print(filtered_reports[7]['appealCode'])
print(filtered_reports[7]['header'])

['SITUATION ANALYSIS Description of the crisis Zambia is undergoing one of the driest agricultural seasons in more than forty years, causing major crop and livestock losses and severely affecting the wellbeing and livelihoods of communities nationwide.', 'According to ongoing reports from the UN, 84 out of 116 districts in the country have been affected by this crisis.', 'The IPC report from August 20231 projected an estimated 58,000 people, between October 2023 and March 2024, to be in an Emergency condition IPC Phase 4 and two million people in Crisis IPC Phase 3 and requiring urgent humanitarian support.', 'On 29 February 2024, the President of Zambia declared a national emergency due to the prolonged drought.', 'On 16 April 2024, the joint rapid needs assessment 2 was commissioned by the Agriculture and Food Security Cluster and the National Government Drought Response Appeal indicated that 6.6 million people needed urgent humanitarian assistance 33 per cent of Zambias total popula

In [21]:
dict_7 = [
    {
        "Hazard": "Drought",
        "Country": "Zambia",
        "Locations": ["Lusaka, Luapula, and the Western, Southern, Central, and Northwestern Provinces", "Western, Southern, and NorthWestern."
                     ],
        "Start_Date": "29 February 2024",
        "End_Date": "NULL",
    }
]
df_7 = pd.DataFrame(dict_7)
df_7['appealCode'] = filtered_reports[7]['appealCode']
df_7['Country'] = [country_name_to_iso3(cntr) for cntr in df_7['Country']]

In [22]:
#print(filtered_reports[8]['appealCode'])
print(filtered_reports[8]['header'])

['Appeal MDRUG050 Total DREF Allocation CHF 479,715 Crisis Category Yellow Hazard Flood Glide Number People Affected 69,283 people People Targeted 19,098 people Event Onset Slow Operation Start Date 22052024 New Operational End Date 30112024 Total Operating Timeframe 6 months Reporting Timeframe Start Date Reporting Timeframe End Date Additional Allocation Requested 157,941 Targeted Areas Central Region, Eastern Region, Western Region Page 1 19 Description of the Event Map of Uganda showing flood affected districts Date when the trigger was met 21082024 What happened, where and when?', 'In April 2024, the Eastern UgandaElgon region experienced heavy rainfall, as forecasted by the Uganda National Meteorological Authority UNMA.', 'This resulted in significant impacts from episodic floods, hailstorms, and landslides in various areas, including Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa.', 'A total of 18,323 people were affected, including thousands of 

In [23]:
dict_8 = [
    {
        "Hazard": "Flood",
        "Country": "Uganda",
        "Locations": ["Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa", 
                      "Manafwa, Lwakhakha, Sironko, Mpologoma, Awoja, Nbuyonga, and Namatala"],
        "Start_Date": "April 2024",
        "End_Date": "31st August 2024",
    }, 
    {
        "Hazard": "Storm",
        "Country": "Uganda",
        "Locations": ["Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa"],
        "Start_Date": "NULL",
        "End_Date": "NULL",
    }, 
    {
        "Hazard": "Mass movement",
        "Country": "Uganda",
        "Locations": ["Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa"],
        "Start_Date": "NULL",
        "End_Date": "NULL",
    }
]
df_8 = pd.DataFrame(dict_8)
df_8['appealCode'] = filtered_reports[8]['appealCode']
df_8['Country'] = [country_name_to_iso3(cntr) for cntr in df_8['Country']]

In [24]:
#print(filtered_reports[9]['appealCode'])
print(filtered_reports[9]['header'])

['SITUATION ANALYSIS Description of the crisis Mozambique is currently experiencing severe effects from the strong 20232024 El Nio season which brought below average rainfall to southern and central Mozambique and aboveaverage rainfall to the northern regions, severely impacting agriculture and rural livelihoods.', 'Additionally, Tropical Storm Filipo in March 2024 impacted 153,000 people, caused significant infrastructural damage, and further devastated agricultural lands, particularly in regions still reeling from the extensive destruction caused by TC Freddy in 2023 OCHA.', 'The compounded effects of these events have severely strained access to basic services and hindered recovery efforts OCHA.', 'Provinces such as Tete, Gaza, Manica, and Inhambane, known for high production and pastoral activities, have seen significant reductions in agricultural output with well belowaverage harvests compared to last year and the fiveyear average.', 'As of April 2024, approximately 690,000 hectar

In [25]:
dict_9 = [
    {
        "Hazard": "Storm", #Freddy
        "Country": "Mozambique",
        "Locations": [],
        "Start_Date": "March 2024",
        "End_Date": "NULL",
    }, 
    {
        "Hazard": "Storm", #Filippo
        "Country": "Mozambique",
        "Locations": [],
        "Start_Date": "2023",
        "End_Date": "NULL",
    }, 
    {
        "Hazard": "Drought",
        "Country": "Mozambique",
        "Locations": ["central and northern zones"],
        "Start_Date": "May 2024",
        "End_Date": "June 2024",
    }
]
df_9 = pd.DataFrame(dict_9)
df_9['appealCode'] = filtered_reports[9]['appealCode']
df_9['Country'] = [country_name_to_iso3(cntr) for cntr in df_9['Country']]

In [27]:
df_combined_examples = pd.concat([df_0, df_1, df_2, df_3, df_4, df_5, df_6, df_7, df_8, df_9], axis=0)
df_combined_examples.to_csv(file_path_save+'labelled_example_haz-type-emdat.csv', index=False)